# Spam Email Classifier

Classify emails/SMS messages as spam or ham (not spam) using TF-IDF features and Naive Bayes / Logistic Regression.

**Suggested real dataset (Kaggle):** search for `SMS Spam Collection Dataset`. Download it and place the CSV as `data/spam.csv` (columns: `label`, `text`) next to this notebook to use real data automatically.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
np.random.seed(42)


## 2. Load Data

In [ ]:
import os

DATA_PATH = "data/spam.csv"

SPAM_TEMPLATES = [
    "WIN a free {prize} now, click here to claim your prize!!!",
    "Congratulations! You've been selected for a free {prize}. Reply YES.",
    "URGENT: your account will be suspended, verify now at {url}",
    "Limited time offer! Buy {prize} at 90% discount, click {url}",
    "You have won ${amount}! Claim your cash prize immediately.",
    "Free entry into our {amount} lottery, text WIN to 8888",
    "Cheap loans available, no credit check, apply now at {url}",
    "Hot singles in your area want to chat with you now!",
    "Your package could not be delivered, confirm details at {url}",
    "Get rich quick with this one simple trick, click {url}",
]
HAM_TEMPLATES = [
    "Hey, are we still on for lunch tomorrow?",
    "Can you send me the report before the meeting?",
    "Thanks for your help earlier, really appreciate it.",
    "Don't forget to pick up milk on your way home.",
    "The project deadline has been moved to next Friday.",
    "Happy birthday! Hope you have a great day.",
    "Let's catch up this weekend, it's been a while.",
    "I'll be running about 10 minutes late for the call.",
    "Great job on the presentation today, well done.",
    "Can we reschedule our meeting to 3pm instead?",
]

def make_synthetic_texts(n=1200):
    rows = []
    for _ in range(n):
        is_spam = np.random.rand() < 0.35
        if is_spam:
            t = np.random.choice(SPAM_TEMPLATES)
            t = t.format(prize=np.random.choice(['iPhone','vacation','laptop','gift card']),
                         url='http://bit.ly/xyz' + str(np.random.randint(100,999)),
                         amount=np.random.randint(100,5000))
            rows.append((t, 'spam'))
        else:
            t = np.random.choice(HAM_TEMPLATES)
            rows.append((t, 'ham'))
    return pd.DataFrame(rows, columns=['text', 'label'])

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH, encoding='latin-1')
    df = df.rename(columns={df.columns[0]: 'label', df.columns[1]: 'text'})[['label', 'text']]
    print(f"Loaded real dataset from {DATA_PATH}, shape={df.shape}")
else:
    print("No local dataset found at 'data/spam.csv'.")
    print("To use real data: search Kaggle for 'SMS Spam Collection Dataset',")
    print("download it, place the CSV at 'spam.csv' inside a 'data/' folder, and re-run this cell.")
    print("Generating a realistic synthetic dataset instead...\n")
    df = make_synthetic_texts()
    print(f"Synthetic dataset generated, shape={df.shape}")

df.head()


## 3. Exploratory Data Analysis

In [ ]:
print(df['label'].value_counts())
plt.figure(figsize=(5,4))
sns.countplot(x='label', data=df)
plt.title('Spam vs Ham distribution')
plt.show()

df['text_length'] = df['text'].str.len()
plt.figure(figsize=(6,4))
sns.histplot(data=df, x='text_length', hue='label', bins=30, kde=True)
plt.title('Message length by class')
plt.show()


## 4. Preprocessing & Train/Test Split

In [ ]:
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['text'], df['label_num'], test_size=0.2, random_state=42, stratify=df['label_num']
)

vectorizer = TfidfVectorizer(stop_words='english', max_features=3000)
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)


## 5. Model Training & Comparison

In [ ]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000)
}

results = {}
preds_dict = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    preds_dict[name] = preds
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    results[name] = {"Accuracy": acc, "F1": f1}
    print(f"{name:20s} | Accuracy: {acc:.3f} | F1: {f1:.3f}")

best_name = max(results, key=lambda k: results[k]["F1"])
best_preds = preds_dict[best_name]
print(f"\nBest model: {best_name}")
print()
print(classification_report(y_test, best_preds, target_names=['ham', 'spam']))


## 6. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, best_preds)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['ham','spam'], yticklabels=['ham','spam'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_name}')
plt.show()


## 7. Try It Yourself

In [ ]:
def predict_message(text, model_name=best_name):
    model = models[model_name]
    vec = vectorizer.transform([text])
    pred = model.predict(vec)[0]
    return 'spam' if pred == 1 else 'ham'

sample = "Congratulations! You have won a free iPhone, click here now!"
print(f"'{sample}' -> {predict_message(sample)}")

sample2 = "Hey, are we still meeting for coffee tomorrow?"
print(f"'{sample2}' -> {predict_message(sample2)}")


## 8. Conclusion

- TF-IDF + Naive Bayes is a strong, fast baseline for spam detection.
- Logistic Regression on TF-IDF features often performs comparably or better.
- Next steps: try n-grams, SMOTE for class imbalance, or transformer-based embeddings once real data is loaded.